# Testing the VocBulwark `vocoder-large` in place of HiFi-GAN

You spotted the channel mismatch (80 vs 96). That is real, but it is the *smallest* of three
differences. Before writing any adapter code, it is worth being precise about all of them:

| | Matcha + HiFi-GAN | `mlr2000/vocoder-large` |
|---|---|---|
| mel bands | 80 | **96** |
| audio rate the mel is computed on | 22050 Hz | **24000 Hz** |
| frames for a 9.66 s clip | 831 | **905** |
| value range (LJ001-0001) | `[-11.51, +1.47]` | `[-0.33, +1.67]` |
| scale | natural log, `fmax=8000`, `center=False` | **Whisper log10, range-clipped and normalised** |
| conditioning | none (single speaker) | **768-d speaker embedding, required** |
| output rate | 22050 Hz | 24000 Hz |

The numbers above are measured, not quoted from the model card — the comparison cell below
reproduces them.

**The consequence:** this is not a drop-in replacement. Matcha's decoder was trained to predict
mels in the left-hand feature space; the vocoder reads the right-hand one. Padding 80 -> 96
does not convert between them, and neither does rescaling. To actually drive this vocoder from
Matcha you would have to **retrain Matcha** with `n_mels=96` and this vocoder's Whisper-style
front-end at 24 kHz.

So what this notebook does is the test that is actually meaningful today:

1. **Copy-synthesis** — real audio -> the vocoder's own mel -> waveform. This measures the
   vocoder's quality ceiling and proves the pipeline works end to end.
2. **A side-by-side mel comparison**, so the mismatch is concrete rather than abstract.
3. **The naive 80 -> 96 pad**, so you can hear exactly why it does not work.

Two things to know before using any output: the model **always embeds a fixed 50-bit
watermark** (it cannot be disabled), and it is **CC-BY-4.0**, so attribution is required.

## Setup

This model's remote code targets **transformers 4.x** and never calls `self.post_init()`.
transformers 5.x reads `all_tied_weights_keys` during `from_pretrained`, which `post_init()` is
what sets, so v5 fails with:

```
AttributeError: 'HiFiGANArchitecture' object has no attribute 'all_tied_weights_keys'
```

Pin below 5 (verified: 4.57.3 and 4.57.6 work, 5.17.0 fails):

```bash
uv run --with 'transformers<5' --extra rocm jupyter lab
```

or permanently: `uv add 'transformers<5'`.

In [1]:
from pathlib import Path

SPK_REPO = "mlr2000/vocoder-large-speaker-encoder"     # produces the 768-d embedding
GEN_REPO = "mlr2000/vocoder-large"                     # the vocoder under test
DET_REPO = "mlr2000/vocoder-large-watermark-detector"  # optional watermark check

TGT_SR      = 24000          # vocoder output rate
MAX_SECONDS = 16             # Whisper's extractor uses a fixed window; keep clips under this
MAX_SAMPLES = MAX_SECONDS * TGT_SR
N_CLIPS     = 3

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "matcha").is_dir())
WAV_DIR   = REPO_ROOT / "data" / "LJSpeech-1.1" / "wavs"
OUT_DIR   = REPO_ROOT / "synth_output" / "vocbulwark"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("repo:", REPO_ROOT)
print("wavs:", WAV_DIR, "->", "found" if WAV_DIR.is_dir() else "MISSING")

repo: /Users/felix/PhD/02_Edu/Matcha-TTS
wavs: /Users/felix/PhD/02_Edu/Matcha-TTS/data/LJSpeech-1.1/wavs -> found


In [3]:
import numpy as np
import soundfile as sf
import torch
import torchaudio.functional as AF
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from transformers import AutoModel, WhisperFeatureExtractor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # ROCm also reports as "cuda"
print("torch", torch.__version__, "| device:", DEVICE)

torch 2.8.0 | device: cpu


## Load the models

`trust_remote_code=True` executes Python fetched from the model repo. That is how the authors
ship the architecture, but review `modeling_hifigan.py` / `bigvgan_model.py` on the Hub first if
running unreviewed code matters on your cluster.

In [ ]:
enc = AutoModel.from_pretrained(SPK_REPO, trust_remote_code=True).eval().to(DEVICE)
gen = AutoModel.from_pretrained(GEN_REPO, trust_remote_code=True).eval().to(DEVICE)

RAW_SR = enc.config.raw_sample_rate          # rate the speaker encoder expects (22050)
N_MELS = gen.config.hifigan_in_channels      # 96
N_FFT  = gen.config.n_fft                    # 1024
HOP    = gen.config.hop_length               # 256

print(f"vocoder  : {N_MELS} mels, n_fft={N_FFT}, hop={HOP} -> {TGT_SR} Hz")
print(f"speaker  : {enc.config.embedding_size}-d embedding, reference audio @ {RAW_SR} Hz")
print(f"watermark: {len(gen.config.fixed_watermark)} bits, embedded automatically, not removable")

## The vocoder's mel front-end

This is the part that is easiest to get subtly wrong. It is a **Whisper** feature extractor, not
the HiFi-GAN-style mel Matcha uses. Two details that matter:

- it runs on audio resampled to **24 kHz**, not 22.05 kHz;
- by default it pads or truncates every input to a fixed 30 s window, which would wreck short
  clips, so `n_samples` and `chunk_length` are overridden below.

In [ ]:
mel_fe = WhisperFeatureExtractor(
    sampling_rate=TGT_SR, n_fft=N_FFT, feature_size=N_MELS, hop_length=HOP
)
mel_fe.n_samples    = MAX_SAMPLES
mel_fe.chunk_length = MAX_SAMPLES / TGT_SR


# 1-D waveform at 24 kHz -> [1, 96, frames] in the vocoder's feature space.
def vocoder_mel(wav_24k):
    feats = mel_fe(wav_24k.cpu().numpy(), sampling_rate=TGT_SR,
                   padding="longest", return_tensors="pt")["input_features"]
    return feats.to(DEVICE)


# A wav on disk -> (original waveform, its rate, speaker reference @ RAW_SR, content @ 24 kHz).
def load_clip(path):
    w, sr = sf.read(str(path), dtype="float32")
    w = torch.tensor(w)
    if w.dim() == 2:
        w = w.mean(1)                                        # to mono
    raw = AF.resample(w, sr, RAW_SR) if sr != RAW_SR else w  # speaker reference
    tgt = AF.resample(raw, RAW_SR, TGT_SR)                   # content for the mel
    return w, sr, raw.unsqueeze(0).to(DEVICE), tgt

## Test 1 — copy-synthesis

Reconstruct real speech through the vocoder. Each clip supplies both the content (its mel) and
the speaker reference (its embedding). This is the vocoder's quality ceiling: Matcha driving it
could only ever sound worse, never better — so if this does not beat your HiFi-GAN baseline,
nothing downstream will.

In [ ]:
results = {}
clips = sorted(WAV_DIR.glob("*.wav"))[:N_CLIPS]
assert clips, f"no wavs under {WAV_DIR}"

for path in clips:
    orig, orig_sr, raw, tgt = load_clip(path)
    mel = vocoder_mel(tgt)
    with torch.no_grad():
        emb   = enc.embed(raw)                                       # [1, 768]
        audio = gen(mel_spectrogram=mel, speaker_embedding=emb).audio.squeeze(1)
    wav = audio[0].float().cpu().numpy()
    results[path.name] = dict(orig=orig, orig_sr=orig_sr, mel=mel, emb=emb, wav=wav)

    sf.write(OUT_DIR / f"{path.stem}_vocbulwark.wav", wav, TGT_SR)
    print(f"{path.name}: mel {tuple(mel.shape)} -> {wav.shape[0]} samples "
          f"({wav.shape[0] / TGT_SR:.2f}s @ {TGT_SR} Hz)")
    print("  original:")
    display(Audio(orig.numpy(), rate=orig_sr))
    print("  vocoded:")
    display(Audio(wav, rate=TGT_SR))

## Test 2 — the two mel spaces, side by side

This reproduces the table at the top from your own data. Note that the frame counts differ for
the *same clip* purely because of 22.05 vs 24 kHz, so even the time axes do not line up.

In [ ]:
from matcha.utils.audio import mel_spectrogram

name = clips[0].name
orig, orig_sr, raw, tgt = load_clip(clips[0])

m_voc = vocoder_mel(tgt)[0].cpu()
m_mat = mel_spectrogram(orig.unsqueeze(0), 1024, 80, 22050, 256, 1024, 0, 8000, center=False)[0]

for label, m, sr in [("Matcha / HiFi-GAN", m_mat, 22050), ("VocBulwark", m_voc, TGT_SR)]:
    print(f"{label:20} bands={m.shape[0]:3d}  frames={m.shape[1]:4d}  "
          f"range=[{m.min():+7.3f}, {m.max():+6.3f}]  mean={m.mean():+7.3f}   (mel @ {sr} Hz)")

fig, axes = plt.subplots(2, 1, figsize=(11, 5.5), constrained_layout=True)
panels = [("Matcha: 80 bands, natural log, fmax=8000", m_mat),
          ("VocBulwark: 96 bands, Whisper log10 normalised", m_voc)]
for ax, (title, m) in zip(axes, panels):
    im = ax.imshow(m, aspect="auto", origin="lower", interpolation="nearest")
    ax.set_title(f"{title}   ({name})")
    ax.set_ylabel("mel band")
    fig.colorbar(im, ax=ax)
axes[-1].set_xlabel("frame")
plt.show()

## Test 3 — why padding 80 -> 96 does not work

The tempting shortcut: keep Matcha's mel, pad 16 zero rows, feed it in. Listen once.

It fails for reasons padding cannot address — the values live on a different scale (natural log
centred near -5 vs normalised log10 centred near +0.25), the 16 appended rows are not "missing
high frequencies" but a different filterbank entirely, and the frame rate corresponds to
22.05 kHz rather than 24 kHz. A linear rescale fixes none of those.

In [ ]:
emb = results[name]["emb"]

padded = torch.zeros(1, N_MELS, m_mat.shape[-1])
padded[:, : m_mat.shape[0]] = m_mat

with torch.no_grad():
    bad = gen(mel_spectrogram=padded.to(DEVICE), speaker_embedding=emb).audio.squeeze(1)
bad_wav = bad[0].float().cpu().numpy()
sf.write(OUT_DIR / f"{clips[0].stem}_naive_pad.wav", bad_wav, TGT_SR)

print("naive 80->96 zero-pad, fed straight to the vocoder:")
display(Audio(bad_wav, rate=TGT_SR))
print("the correct copy-synthesis of the same clip, for reference:")
display(Audio(results[name]["wav"], rate=TGT_SR))

## Optional — watermark detector

Every waveform this vocoder produces carries a fixed 50-bit watermark. Worth confirming if you
intend to publish or evaluate the audio, since it is a deliberate modification of the signal.

In [ ]:
det = AutoModel.from_pretrained(DET_REPO, trust_remote_code=True).eval()

with torch.no_grad():
    r = det.detect(torch.tensor(results[name]["wav"])[None])
    print(f"vocoded: {r['matches']}/{r['n_bits']} bits  p={r['p_value']:.1e}")
    r = det.detect(torch.randn(1, TGT_SR))
    print(f"noise  : {r['matches']}/{r['n_bits']} bits  p={r['p_value']:.1e}   (control)")

## Optional — your Matcha checkpoint, for an A/B

The existing HiFi-GAN path, unchanged, so you can compare it against the copy-synthesis above.
This is *not* the new vocoder driven by Matcha — as established, that needs retraining. Point
`MATCHA_CHECKPOINT` at a run of your own.

In [ ]:
MATCHA_CHECKPOINT  = REPO_ROOT / "logs/train/ljspeech/runs/2026-09-16_15-20-25/checkpoints/last.ckpt"

from matcha.utils.utils import get_user_data_dir

HIFIGAN_CHECKPOINT = get_user_data_dir() / "hifigan_T2_v1"

if not Path(MATCHA_CHECKPOINT).is_file():
    print(f"skipped: no Matcha checkpoint at {MATCHA_CHECKPOINT}")
elif not Path(HIFIGAN_CHECKPOINT).is_file():
    print(f"skipped: no HiFi-GAN checkpoint at {HIFIGAN_CHECKPOINT}")
    print("         download it once with:  matcha-tts --text hi")
else:
    from matcha.cli import load_matcha, load_vocoder, to_waveform
    from matcha.text import text_to_sequence
    from matcha.utils.utils import intersperse

    model = load_matcha("matcha_ljspeech", MATCHA_CHECKPOINT, DEVICE)
    vocoder, denoiser = load_vocoder("hifigan_T2_v1", HIFIGAN_CHECKPOINT, DEVICE)

    text = "The sound of a neural vocoder depends on the spectrogram it was trained to read."
    x = torch.tensor(intersperse(text_to_sequence(text, ["english_cleaners2"])[0], 0),
                     dtype=torch.long, device=DEVICE)[None]
    x_len = torch.tensor([x.shape[-1]], dtype=torch.long, device=DEVICE)

    with torch.inference_mode():
        out = model.synthesise(x, x_len, n_timesteps=10, temperature=0.667, length_scale=1.0)
        wav = to_waveform(out["mel"], vocoder, denoiser)

    print(f"Matcha mel {tuple(out['mel'].shape)} -> HiFi-GAN waveform @ 22050 Hz")
    display(Audio(wav.numpy(), rate=22050))

## Where that leaves things

The vocoder works, and copy-synthesis shows what it can do — but it cannot be swapped in behind
your current Matcha checkpoint. Three things would have to change together:

1. **Retrain Matcha with `n_mels=96`**, using this vocoder's Whisper front-end at 24 kHz as the
   target feature. That means changing the mel extraction in
   `matcha/data/text_mel_datamodule.py`, the `n_feats` in `configs/model/matcha.yaml` and the
   data config, then regenerating your data statistics.
2. **Decide what to do about speaker conditioning.** LJSpeech is single-speaker, so the simplest
   route is to compute one embedding from a reference clip and reuse it for every utterance.
3. **Accept the watermark and the 24 kHz output rate.** Both are fixed.

That is a retraining run, not an adapter. Worth confirming from Test 1 that the quality actually
beats your HiFi-GAN baseline before committing to it.